<a href="https://colab.research.google.com/github/Heptazero/ml-course-labs/blob/main/data-mining/elliptic-bitcoin-fraud-detection/01_eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 下载

In [ ]:
!pip install torch_geometric -q
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from torch_geometric.datasets import EllipticBitcoinDataset

DATA_ROOT = Path('/content/ml-course-labs-data/elliptic') if Path('/content').exists() else Path('data/elliptic')
dataset = EllipticBitcoinDataset(root=str(DATA_ROOT))
data = dataset[0]
counts = torch.bincount(data.y, minlength=3)

print(data)
# 中文字体补丁：本地与 Colab 都使用 Python 标准库下载，不依赖 wget
from urllib.request import urlretrieve

import matplotlib.font_manager as fm

_FONT_PATH = Path("NotoSansCJKtc-Regular.otf")
_FONT_URL = (
    "https://raw.githubusercontent.com/notofonts/noto-cjk/main/"
    "Sans/OTF/TraditionalChinese/NotoSansCJKtc-Regular.otf"
)
if not _FONT_PATH.exists():
    try:
        urlretrieve(_FONT_URL, _FONT_PATH)
    except OSError as error:
        print(f"中文字体下载失败：{error}")
if _FONT_PATH.exists():
    fm.fontManager.addfont(_FONT_PATH)
    plt.rcParams["font.family"] = "Noto Sans CJK TC"
plt.rcParams["axes.unicode_minus"] = False

## 展示
- 一个节点就是一笔比特币交易，每笔交易身上挂着165个数字
  - 前94个叫"本地特征"，是这笔交易自己的信息——发生在49个时间步里的第几步、这笔交易有几个输入几个输出、手续费多少、总输出金额多少，还有一些跟它直接相连的输入输出的平均值
  - 剩下72个叫"聚合特征"，是这笔交易周围一圈邻居交易的统计值——邻居们的最大值、最小值、标准差、相关系数。
  - 每一项数值："这一项比平均水平高还是低"

In [ ]:
import pandas as pd

feature_df = pd.DataFrame(data.x.numpy())
print("节点数、特征数：", feature_df.shape)
feature_df.head()

- 每一列是一条边
- 上面那行是边的起点，下面那行是边的终点。

In [ ]:
print("边的数量：", data.edge_index.shape[1])
data.edge_index[:, :5]

In [ ]:
import matplotlib.pyplot as plt

names = ['正常(0)', '欺诈(1)', '未标注(2)']
plt.bar(names, counts.tolist())
plt.title('各类别节点数量')
plt.show()

- 原始论文里每个时间段大概跨两周，49个时间段前后连起来大概覆盖了近三年的比特币交易记录。

In [ ]:

import torch
time_steps = data.x[:, 0]
plt.hist(time_steps.numpy(), bins=49)
plt.title('每个时间步的交易数量')
plt.xlabel('时间步')
plt.show()

In [ ]:
print("特征里有没有缺失值：", torch.isnan(data.x).any().item())